In [1]:

import pandas as pd
from engine import load_env, create_engine_from_env
from models.manufacture_module import get_manufacturer_df, get_active_unique_manufacturers, get_manufacturer_bulletins_json, print_bulletin_details, convert_bulletin_to_df, save_vehicle_models_to_csv, batch_save_manufacturer_models, search_models_by_description
 

In [2]:

import json
from typing import Union, Any

def parse_json_string(json_string: str) -> Union[dict, list]:
    """
    Parse a JSON string into a Python object (dict or list).

    Args:
        json_string (str): A valid JSON string.

    Returns:
        dict or list: Parsed JSON object.

    Raises:
        ValueError: If the input is not valid JSON.
        TypeError: If input is not a string.
    """
    if not isinstance(json_string, str):
        raise TypeError("Input must be a JSON string")

    try:
        return json.loads(json_string)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON string: {exc}") from exc
 

In [3]:

from pathlib import Path
from datetime import datetime

def load_latest_oem_data(manufacturer: str, landing_zone_path: str = None):
    """
    Load the latest OEM data file from the landing zone for a given manufacturer.

    Args:
        manufacturer (str): Manufacturer name (e.g., "hyundai", "mazda", "mitsubishi")
        landing_zone_path (str): Path to landing zone directory.
                                 Defaults to accy_v2/data/landing_zone relative to current dir

    Returns:
        pd.DataFrame: DataFrame loaded from the latest file for the manufacturer

    Raises:
        FileNotFoundError: If manufacturer folder or data files not found
        ValueError: If no supported files found (CSV or XLSX)
    """
    if landing_zone_path is None:
        landing_zone_path = Path("../data/landing_zone")
    else:
        landing_zone_path = Path(landing_zone_path)

    # Normalize manufacturer name (case-insensitive)
    manufacturer_clean = manufacturer.lower().strip()
    manufacturer_dir = landing_zone_path / manufacturer_clean

    # Check if manufacturer directory exists
    if not manufacturer_dir.exists():
        raise FileNotFoundError(
            f"Manufacturer folder not found: {manufacturer_dir}\n"
            f"Available manufacturers: {[d.name for d in landing_zone_path.iterdir() if d.is_dir()]}"
        )

    # Find all data files (CSV or XLSX)
    data_files = []
    for ext in ["*.csv", "*.xlsx", "*.xls"]:
        data_files.extend(manufacturer_dir.glob(ext))

    if not data_files:
        raise ValueError(
            f"No data files (CSV/XLSX) found in {manufacturer_dir}\n"
            f"Files in directory: {list(manufacturer_dir.iterdir())}"
        )

    # Get the latest file by modification time
    latest_file = max(data_files, key=lambda p: p.stat().st_mtime)
    mod_time = datetime.fromtimestamp(latest_file.stat().st_mtime)

    print(f"[OK] Loading {manufacturer.upper()} data")
    print(f"     File: {latest_file.name}")
    print(f"     Modified: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")

    # Load based on file type
    try:
        if latest_file.suffix.lower() == ".csv":
            df = pd.read_csv(latest_file)
        elif latest_file.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(latest_file)
        else:
            raise ValueError(f"Unsupported file type: {latest_file.suffix}")

        print(f"     Rows: {len(df)}, Columns: {len(df.columns)}")
        return df

    except Exception as e:
        raise RuntimeError(f"Failed to load file {latest_file.name}: {str(e)}")



In [4]:

# Loaded environment variables and created database engine
load_env()
engine = create_engine_from_env()


python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 8


In [5]:

# # # Batch save vehicle models for specified manufacturers
# manufactures = ["Hyundai","Honda","Kia","Mazda","Genesis", "Mitsubishi", "Volkswagen"]

# # batch_save_manufacturer_models(engine, manufactures)


In [6]:

model_number_db = pd.read_csv("db/db_vehicle_models.csv")



In [7]:
from pathlib import Path
from datetime import datetime

def load_latest_oem_data(
    manufacturer: str, 
    landing_zone_path: str = None, 
    header_row: int = None,
    auto_detect_header: bool = True,
    header_keywords: list = None
):
    """
    Load the latest OEM data file from the landing zone for a given manufacturer.

    Args:
        manufacturer (str): Manufacturer name (e.g., "hyundai", "mazda", "mitsubishi")
        landing_zone_path (str): Path to landing zone directory.
                                 Defaults to accy_v2/data/landing_zone relative to current dir
        header_row (int): Specific row index to use as headers (0-based). 
                         If None and auto_detect_header=True, will auto-detect.
                         If None and auto_detect_header=False, defaults to 0 (first row).
        auto_detect_header (bool): If True, searches for row containing header keywords.
        header_keywords (list): Keywords to search for in rows (e.g., ["Year", "Model", "Trim"]).
                               Defaults to ["Year", "Model", "Trim", "Make", "Manufacturer"]

    Returns:
        pd.DataFrame: DataFrame loaded from the latest file for the manufacturer

    Raises:
        FileNotFoundError: If manufacturer folder or data files not found
        ValueError: If no supported files found (CSV or XLSX)
    """
    if landing_zone_path is None:
        landing_zone_path = Path("../data/landing_zone")
    else:
        landing_zone_path = Path(landing_zone_path)

    # Default header keywords to search for
    if header_keywords is None:
        header_keywords = ["Year", "Model", "Trim"]

    # Normalize manufacturer name (case-insensitive)
    manufacturer_clean = manufacturer.lower().strip()
    manufacturer_dir = landing_zone_path / manufacturer_clean

    # Check if manufacturer directory exists
    if not manufacturer_dir.exists():
        raise FileNotFoundError(
            f"Manufacturer folder not found: {manufacturer_dir}\n"
            f"Available manufacturers: {[d.name for d in landing_zone_path.iterdir() if d.is_dir()]}"
        )

    # Find all data files (CSV or XLSX)
    data_files = []
    for ext in ["*.csv", "*.xlsx", "*.xls"]:
        data_files.extend(manufacturer_dir.glob(ext))

    if not data_files:
        raise ValueError(
            f"No data files (CSV/XLSX) found in {manufacturer_dir}\n"
            f"Files in directory: {list(manufacturer_dir.iterdir())}"
        )

    # Get the latest file by modification time
    latest_file = max(data_files, key=lambda p: p.stat().st_mtime)
    mod_time = datetime.fromtimestamp(latest_file.stat().st_mtime)

    print(f"[OK] Loading {manufacturer.upper()} data")
    print(f"     File: {latest_file.name}")
    print(f"     Modified: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")

    # Load based on file type
    try:
        if latest_file.suffix.lower() == ".csv":
            df_raw = pd.read_csv(latest_file, header=None)
        elif latest_file.suffix.lower() in [".xlsx", ".xls"]:
            df_raw = pd.read_excel(latest_file, header=None)
        else:
            raise ValueError(f"Unsupported file type: {latest_file.suffix}")

        # Auto-detect header row if requested
        if header_row is None and auto_detect_header:
            header_row = _find_header_row(df_raw, header_keywords)
            if header_row is None:
                print(f"     [WARNING] Could not auto-detect header row. Using row 0.")
                header_row = 0
            else:
                print(f"     [Auto-detected] Header row: {header_row}")
        elif header_row is None:
            header_row = 0
        
        # Apply header row
        if header_row > 0:
            df_raw.columns = df_raw.iloc[header_row]
            df = df_raw.iloc[header_row + 1:].reset_index(drop=True)
        else:
            df = df_raw.copy()
            if header_row == 0:
                df.columns = df.iloc[0]
                df = df.iloc[1:].reset_index(drop=True)

        print(f"     Rows: {len(df)}, Columns: {len(df.columns)}")
        print(f"     Column names: {list(df.columns[:5])}{'...' if len(df.columns) > 5 else ''}")
        return df

    except Exception as e:
        raise RuntimeError(f"Failed to load file {latest_file.name}: {str(e)}")


def _find_header_row(df: pd.DataFrame, keywords: list) -> int:
    """
    Find the row index that contains most of the header keywords.
    
    Args:
        df (pd.DataFrame): DataFrame loaded without headers (all rows are data)
        keywords (list): List of keywords to search for (e.g., ["Year", "Model", "Trim"])
    
    Returns:
        int: Row index of detected header row, or None if not found
    """
    keywords_lower = [kw.lower() for kw in keywords]
    best_row = None
    best_score = 0
    
    # Search first 10 rows for headers
    for row_idx in range(min(10, len(df))):
        row_values = [str(val).lower() for val in df.iloc[row_idx]]
        
        # Count how many keywords match in this row
        matches = sum(1 for kw in keywords_lower if any(kw in val for val in row_values))
        
        if matches > best_score:
            best_score = matches
            best_row = row_idx
    
    # Only return if we found at least 2 matching keywords
    return best_row if best_score >= 2 else None
      

In [8]:
model_number_db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Description   395 non-null    object
 1   Drivetrain    395 non-null    object
 2   Manufacturer  395 non-null    object
 3   ModelName     395 non-null    object
 4   ModelNumber   395 non-null    object
 5   ModelYear     395 non-null    int64 
 6   Package       395 non-null    int64 
 7   PassDoors     395 non-null    int64 
 8   TrimName      385 non-null    object
 9   engine_type   123 non-null    object
dtypes: int64(3), object(7)
memory usage: 31.0+ KB


In [9]:

# Now you can load the latest OEM data for any manufacturer
hyundai_raw_df = load_latest_oem_data("hyundai")
# print(f"\nHyundai data loaded successfully!")
# print(f"Columns: {list(hyundai_df.columns)}")
# print(f"First few rows:")
# hyundai_raw_df.head()
 

[OK] Loading HYUNDAI data
     File: 2026-8-1 HACC MAF DIST - 08102026.xlsx
     Modified: 2026-08-11 10:38:38
     [Auto-detected] Header row: 1
     Rows: 5088, Columns: 33
     Column names: ['Model Year From', 'Model Year To ', 'Model', 'Trim 1', 'Trim 2 ']...


c:\Users\paxm\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [13]:
hyundai_raw_df.columns

Index(['Model Year From', 'Model Year To ', 'Model', 'Trim 1', 'Trim 2 ',
       'Trim 3', 'Trim 4 ', 'Trim 5', 'Trim 6', 'Trim 7', 'Trim 8',
       'Accessory Category', 'Accessory Group ', 'Accessory Description',
       'Accessory Category (FR)', 'Accessory Group (FR)',
       'Accessory Description (FR)', 'Part Number', 'DNET', 'MSRP',
       'Labour Rate ', 'Suggested Labour Hours',
       'Suggested Base Retail Price w labour ', 'Yearly Rate', 'Weekly Rate',
       '#of Payments', 'Weekly Payment', 'PKG Code', 'Comments',
       'Comments (FR)', 'New Part SOS', 'Outgoing Part Flag', 'MAF Update'],
      dtype='object', name=1)

In [14]:
hyundai_raw_df_filtered = hyundai_raw_df[
                                        (hyundai_raw_df['Model Year From'].notnull()) 
                                        & (hyundai_raw_df['Model'].notnull()) 
                                    ][["Model Year From", "Model"]]

In [15]:

# I want to check it there any models that are in hyundai_raw_df_filtered that are not in model_number_db, and if so, I want to print them out.
hyundai_models = set(hyundai_raw_df_filtered["Model"].str.lower().unique())
model_number_models = set(model_number_db["ModelName"].str.lower().unique())
missing_models = hyundai_models - model_number_models
print("Models in hyundai_raw_df_filtered but not in model_number_db:")
print(missing_models)
 

Models in hyundai_raw_df_filtered but not in model_number_db:
{'kona ev', 'gv70 ev', 'g80 ev'}


In [16]:

model_number_db["ModelName"].unique().tolist()
 

['Electrified G80',
 'Electrified GV70',
 'G70',
 'G80',
 'G90',
 'GV60',
 'GV70',
 'GV80',
 'GV80 Coupe',
 'Elantra',
 'Elantra Hybrid',
 'Elantra N',
 'IONIQ 5',
 'IONIQ 6',
 'Kona',
 'Kona Electric',
 'NEXO',
 'Palisade',
 'Santa Cruz',
 'Santa Fe',
 'Santa Fe Hybrid',
 'Sonata',
 'Tucson',
 'Tucson Hybrid',
 'Tucson Plug-In Hybrid',
 'Venue',
 'IONIQ 5 N',
 'IONIQ 9',
 'Palisade Hybrid',
 'Sonata Hybrid',
 'Eclipse Cross',
 'Mirage',
 'Outlander',
 'Outlander PHEV',
 'RVR',
 'Outlander Plug-In Hybrid']

In [17]:
model_number_db[
                (1==1)
                & (model_number_db["ModelYear"]==2024)
                &(model_number_db["ModelName"].str.contains("G80"))
                # & (model_number_db["engine_type"].str.contains("ele"))
                ][["ModelName", "ModelYear", "TrimName", "Description", "ModelNumber", "engine_type"]]

,ModelName,ModelYear,TrimName,Description,ModelNumber,engine_type
0,Electrified G80,2024,NaN,Awd,G8ES4ZE1GP00,electric
6,G80,2024,2.5T Advanced,2.5t Advanced Awd,G8CS4K2DGA00,2.5t
7,G80,2024,3.5T Sport Plus,3.5t Sport Plus Awd,G8CS4K3BGSAU,3.5t


In [18]:

# model_number_db[model_number_db["ModelName"]=="GV70"].value_counts(by=["ModelYear"])

model_number_db.loc[
    model_number_db["ModelName"]=="GV70",
    "ModelYear"
    ].value_counts()
 

ModelYear
2024    6
2025    5
2026    5
Name: count, dtype: int64

In [19]:
model_number_db.columns

Index(['Description', 'Drivetrain', 'Manufacturer', 'ModelName', 'ModelNumber',
       'ModelYear', 'Package', 'PassDoors', 'TrimName', 'engine_type'],
      dtype='object')

In [20]:
make_df_genesis = model_number_db[model_number_db["Manufacturer"].str.contains("Genesis", case=False)]
make_df_hyundai = model_number_db[model_number_db["Manufacturer"].str.contains("Hyundai", case=False)]

In [21]:

make_df_genesis[make_df_genesis["TrimName"].isna()]


,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type
0,Awd,ALL_WHEEL_DRIVE,GENESIS,Electrified G80,G8ES4ZE1GP00,2024,450310,4,NaN,electric
61,Awd *ltd Avail*,ALL_WHEEL_DRIVE,GENESIS,GV60,V6EW5ZE2GW00,2026,481701,4,NaN,NaN


In [31]:
make_df_hyundai.columns

Index(['Description', 'Drivetrain', 'Manufacturer', 'ModelName', 'ModelNumber',
       'ModelYear', 'Package', 'PassDoors', 'TrimName', 'engine_type'],
      dtype='object')

In [82]:

make_df_hyundai[
    (make_df_hyundai["ModelYear"]==2026)
    # (1==1)
    &(make_df_hyundai["ModelName"].str.contains("ioniq 5", case=False, na=False))
    # &(make_df_hyundai["ModelName"].str.contains("fe", case=False, na=False))
    # & (make_df_hyundai["Description"].str.contains("XRT", case=False, na=False))
    # & (make_df_hyundai["TrimName"].str.contains("XRT", case=False, na=False))
    # & (make_df_hyundai["engine_type"].str.contains("Hybrid", case=False, na=False))
    # & (make_df_hyundai["Package"]=="KE00")
    ][["ModelYear","ModelNumber", "ModelName", "TrimName", "Description", "engine_type"]]
 

,ModelYear,ModelNumber,ModelName,TrimName,Description,engine_type
229,2026,I5EW5ZE4PRLR,IONIQ 5,Preferred,Preferred Rwd Long Range,NaN
230,2026,I5EW5ZE2PRLR,IONIQ 5,Preferred,Preferred Awd Long Range,NaN
231,2026,I5EW5ZE2PRUP,IONIQ 5,Preferred,Preferred Awd Long Range W/ultimate Package,NaN


In [30]:

make = "Hyundai"
year = 2024
keywords = ['Elantra', "TCR"]


search_models_by_description(make, year, keywords)


""


In [24]:
model_number_db.columns

Index(['Description', 'Drivetrain', 'Manufacturer', 'ModelName', 'ModelNumber',
       'ModelYear', 'Package', 'PassDoors', 'TrimName', 'engine_type'],
      dtype='object')

In [25]:

year = 2024
Manufacturer = "Genesis"
ModelName = 'G90'

model_number_db[
    (1==1)
    & (model_number_db["ModelYear"] == year)
    & (model_number_db["Manufacturer"].str.contains(Manufacturer, case=False))
    & (model_number_db["ModelName"].str.contains(ModelName, case=False, na=False))
    ][["ModelYear", "ModelNumber", "ModelName", "TrimName", "Package", "engine_type","Manufacturer"]]


,ModelYear,ModelNumber,ModelName,TrimName,Package,engine_type,Manufacturer
8,2024,G9CS4K3BXXPS,G90,3.5T electric Prestige,450256,electric,GENESIS


In [26]:
mitsubishi_df = model_number_db[model_number_db["Manufacturer"].str.contains("mitsubishi", case=False, na=False)]

In [27]:
mitsubishi_df.columns

Index(['Description', 'Drivetrain', 'Manufacturer', 'ModelName', 'ModelNumber',
       'ModelYear', 'Package', 'PassDoors', 'TrimName', 'engine_type'],
      dtype='object')

In [28]:

year = 2026
# Manufacturer = "Genesis"
ModelName = 'outlander'
trim = 'es'
Description = 's-awc'

mitsubishi_df[
    (1==1)
    & (mitsubishi_df["ModelYear"] == year)
    # & (mitsubishi_df["Manufacturer"].str.contains(Manufacturer, case=False))
    & (mitsubishi_df["ModelName"].str.contains(ModelName, case=False, na=False))
    & (mitsubishi_df["TrimName"].str.contains(trim, case=False, na=False))
    # & (mitsubishi_df["Description"].str.contains(Description, case=False, na=False))
    ][["ModelYear", "ModelNumber", "ModelName", "TrimName", "Description", "Package", "engine_type","Manufacturer"]]


,ModelYear,ModelNumber,ModelName,TrimName,Description,Package,engine_type,Manufacturer
363,2026,CO45-B,Outlander,ES,Es Awc,481877,NaN,MITSUBISHI
370,2026,COEV-B,Outlander Plug-In Hybrid,ES,Es Awc,483107,NaN,MITSUBISHI


In [29]:
make = "mitsubishi"
year = 2026
keywords =  ['outlander', 'phev', 'gt']
keywords =  ['outlander', 'es', 's-awc']



search_models_by_description(make, year, keywords)


,Description,Drivetrain,Manufacturer,ModelName,ModelNumber,ModelYear,Package,PassDoors,TrimName,engine_type
363,Es Awc,ALL_WHEEL_DRIVE,MITSUBISHI,Outlander,CO45-B,2026,481877,4,ES,NaN
370,Es Awc,ALL_WHEEL_DRIVE,MITSUBISHI,Outlander Plug-In Hybrid,COEV-B,2026,483107,4,ES,NaN
